# Проект по Анализу Данных

### Дрим Тим:
1. Гусев Богдан

2. Андреев Михаил

3. Просвирина Мария

4. Келесиди Николай

5. Ковырялина Мария

6. Землянская Евгения

### Описание Проекта
1. Описание датасета

Классификатор строительных ресурсов (КСР) — это систематизированный перечень используемых при строительстве материалов, изделий, конструкций, оборудования, машин и механизмов. Классификатор размещается в ФГИС ЦС и применяется для создания единой базы кодов ресурсов, а также для мониторинга их стоимости. Каждый ресурс получает уникальный код, гармонизированный с Общероссийским классификатором продукции по видам экономической деятельности (ОКПД2).

Формат данных:

Доступен для скачивания в виде Excel‑файлов с возможностью фильтрации и сортировки прямо в ФГИС ЦС.

В примере Yandex Disk представлены колонки:

record_name — наименование ресурса

record_name_2 — аналогичное наименование ресурса

ref_code — код ресурса (КСР)

ref_name — краткое наименование

ref_unit — единица измерения

Полный справочник доступен на портале ФГИС ЦС: https://fgiscs.minstroyrf.ru/ksr/navigate

2. Задание

Выделить параметры для каждого класса/подкласса КСР

При выборе группы (например, «Листы хризотилцементные волнистые» — группа 01.1.01.04) необходимо извлечь из названий ресурсов ключевые характеристики (толщину, профиль, количество волн и т. д.) и на их основе сформировать динамические фильтры.

Предобработка и аналитика

Разбить исходный Excel на три листа:

Материалы
Оборудование
Механизмы
Для каждой записи провести токенизацию наименования по пробелам и регулярным выражениям, подсчитать частотность встречаемых фраз (например, «толщина 5,2 мм», «профиль 40/150») и агрегировать их для последующей фильтрации.

Организовать автодополнение по имени ресурса (по аналогии с Yandex Market): при вводе фрагмента названия пользователю предлагаются подходящие варианты со списком доступных параметров (Например видеокарта и для нее варианты 4гб, 12 гб по объему видеопамяти или фирма-производитель.)

Построить дашборд на Streamlit

Sidebar: выбор группы/класса/подкласса из иерархии КСР

Основная панель:

Список доступных фильтров (чекбоксы/слайдеры) по выделенным характеристикам

Таблица или карточки с отфильтрованными ресурсами и их кодами

Подсчёт и отображение частот использования каждого параметра (бар-чарт или таблица)

Экспорт результатов в CSV

3. Ссылки и представление конечного результата

Пример набора данных (частичный ручной из реальных данных, грязный): https://disk.yandex.ru/d/xUKDvxRmZrfwtQ

Полный справочник КСР: https://fgiscs.minstroyrf.ru/ksr/navigate

Технологии проекта:

Python (pandas, regex, nltk/spacy для токенизации)

Streamlit — быстрое создание интерактивного веб‑интерфейса

4. Презентация проекта

Презентация с основной инфой

Продолжительность: 10-15 минут

Конечный продукт:

Локально запускаемый Streamlit‑приложение, в котором по выбору класса КСР появляется набор динамических фильтров и автодополнение по названию

Возможность выгрузки отобранных ресурсов с их кодами и характеристиками в CSV

Этот проект позволяет получить удобный инструмент для поиска и анализа строительных ресурсов по классификатору КСР, аналогичный принципам фильтрации и автодополнения, применяемым на крупных маркетплейсах.

# Предисловие. Трудностей было много, но мы справлялись! Плакали и справлялись!

### Загружаем все нужные Библиотеки и скачиваем наши данные

In [ ]:
import pandas as pd
import re

In [ ]:
df1= pd.read_excel('Классификатор 19052025.xlsx', sheet_name='Материалы, изд, констр и оборуд', header=None)
df2 = pd.read_excel('Классификатор 19052025.xlsx', sheet_name='Машины и механизмы', header=None)
df = pd.concat([df1, df2], ignore_index=True)

### Выделение кода ОКПД2 и нормализация структуры КСР

In [ ]:
filtered_df = df[df[0].astype(str).str.contains('-', na=False)].rename(columns={0: 'Код ресурса', 1: 'Наименование', 2: 'Ед.изм.'}) # Фильтруем строки, где в первом столбце содержится дефис ('-')
result_df = filtered_df[filtered_df['Код ресурса'].str.count('-') == 2] # Оставляем только те строки, где в 'Коде ресурса' ровно два дефиса
result_df.reset_index(drop=True, inplace=True)
result_df['Код ОКПД2'] = result_df['Код ресурса'].str.slice(0, 12) # Выделение первых 12 символов как 'Код ОКПД2'
result_df['Код ресурса'] = result_df['Код ресурса'].str.slice(13)
result_df

C:\Users\user\AppData\Local\Temp\ipykernel_8068\4129377737.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result_df['Код ОКПД2'] = result_df['Код ресурса'].str.slice(0, 12)
C:\Users\user\AppData\Local\Temp\ipykernel_8068\4129377737.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result_df['Код ресурса'] = result_df['Код ресурса'].str.slice(13)


,Код ресурса,Наименование,Ед.изм.,Код ОКПД2
0,01.1.01.01-0002-000,Детали фасонные коньковые к листам хризотилцем...,100 компл,23.65.12.190
1,01.1.01.02-0011-000,"Доска электротехническая дугостойкая (АЦЭИД), ...",т,23.65.12.190
2,01.1.01.04-1018-000,"Листы хризотилцементные волнистые, профиль 40/...",м2,23.65.12.111
3,01.1.01.04-1022-000,"Листы хризотилцементные волнистые, профиль 40/...",м2,23.65.12.111
4,01.1.01.04-1024-000,"Листы хризотилцементные волнистые, профиль 40/...",м2,23.65.12.111
...,...,...,...,...
97034,96.01.10-001-000,"Трубоукладчики для труб диаметром до 700 мм, г...",шт,28.22.14.159
97035,96.01.15-001-000,"Тракторы на гусеничном ходу, мощность 128,7 кВ...",шт,28.92.50.000
97036,96.01.15-002-000,"Тракторы на гусеничном ходу, мощность 59 кВт (...",шт,28.92.50.000
97037,96.01.17-001-000,Агрегаты для сварки полиэтиленовых труб,шт,27.90.31.110


### Категоризация ресурсов на материалы, оборудование и механизмы по префиксу кода ресурса
На основе префикса кода ресурса производится отнесение каждой записи к одной из категорий: материалы (≤59), оборудование (60–89), механизмы (90 и выше).

In [ ]:
def map_prefix_to_cat(prefix: str) -> str:
    try:
        num = int(prefix)
    except ValueError:
        return 'Unknown'
    if num <= 59:
        return 'Materials'
    elif num <= 89:
        return 'Equipment'
    else:
        return 'Machines'

def split_by_category(df):
    df = df.copy()
    df['category'] = df['Код ресурса'].astype(str).str.split('.').str[0].apply(map_prefix_to_cat) # Получаем числовой префикс из 'Кода ресурса' до первой точки и применяем функцию категоризации


    df['category'].fillna('Unknown', inplace=True) # Заполняем возможные пропуски значением "Unknown"

    result = {} # Создаём словарь с отдельными DataFrame по каждой категории
    for cat in ['Materials', 'Equipment', 'Machines', 'Unknown']:
        result[cat] = df[df['category'] == cat].drop(columns=['category']) # Фильтруем строки по категории и удаляем вспомогательный столбец 'category'
    return result

### Сохранение результатов классификации в файл Excel (по листам)

In [ ]:
splits = split_by_category(result_df) # Разбиваем DataFrame на категории с помощью ранее определённой функции split_by_category

sheet_names = {
    'Materials': 'Материалы',
    'Equipment': 'Оборудование',
    'Machines': 'Механизмы',
}
# Создаем Excel-файл с несколькими листами, по категориям ресурсов
with pd.ExcelWriter('processed_by_category.xlsx', engine='openpyxl') as writer:
    for cat, subdf in splits.items():
        rus_name = sheet_names.get(cat, cat)[:31]
        subdf.to_excel(writer, sheet_name=rus_name, index=False)

print("Результат разбивки сохранён в processed_by_category.xlsx")

C:\Users\user\AppData\Local\Temp\ipykernel_8068\3248530179.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['category'].fillna('Unknown', inplace=True)


Результат разбивки сохранён в processed_by_category.xlsx


### Нормализация текстов: очистка от лишних символов, приведение к нижнему регистру, замена форматов

In [ ]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()  # Приводим к нижнему регистру
    text = re.sub(r'[()<>«»]', '', text)  # Убираем скобки и кавычки
    text = re.sub(r'[,\.;:]', ' ', text)  # Убираем ненужные знаки препинания
    text = re.sub(r'(\d+),(\d+)', r'\1.\2', text)  # Заменяем запятую на точку в числах
    text = re.sub(r'\bmm\b', 'мм', text)  # Приводим "mm" к "мм"
    text = text.strip()  # Убираем лишние пробелы в начале и в конце
    text = re.sub(r'\s+', ' ', text)  # Заменяем множественные пробелы на один
    return text

Строим функцию токенизации

In [ ]:
def tokenize(text):
    tokens = re.findall(r'\b[\w/.-]+\b', text)  # Разбиваем текст на слова, числа и символы
    return tokens

#Теперь начинается самый сложный этап
### Извлечение параметров из текстового описания ресурса
мы вручную отобрали 217 параметров из КСР ☠️☠️☠️, но streamlit не одобрил наши старания(страдания), пришлось уменьшить до 160

In [ ]:
def extract_params(text):
    params = {}  # Словарь для хранения извлечённых параметров

        # Толщина (например, "толщина 5 мм")
    match = re.search(r'толщина\s*(\d+(?:\.\d+)?)\s*мм', text, re.IGNORECASE)
    if match:
        params['thickness_mm'] = float(match.group(1))

    # Профиль (например, "профиль 40/150")
    match = re.search(r'профиль\s*(\d+/\d+)', text, re.IGNORECASE)
    if match:
        params['profile'] = match.group(1)

    # Размеры (например, "размеры 475× 565× 60 мм")
    match = re.search(r'размеры\s*(\d+)\s*×\s*(\d+)\s*×\s*(\d+)\s*мм', text, re.IGNORECASE)
    if match:
        params['dimensions_mm'] = f"{match.group(1)}×{match.group(2)}×{match.group(3)}"

    # Импульсный ток (например, "импульсный ток 25 кА")
    match = re.search(r'импульсный\s*ток\s*(\d+(?:\.\d+)?)\s*к[Аа]', text, re.IGNORECASE)
    if match:
        params['impulse_current_ka'] = float(match.group(1))

    # Уровень напряжения защиты (например, "уровень напряжения защиты 1.5 кВ")
    match = re.search(r'уровень\s*напряжения\s*защиты\s*(\d+[.,]?\d*)\s*к[Вв]', text, re.IGNORECASE)
    if match:
        params['protection_voltage_kv'] = float(match.group(1).replace(',', '.'))

    # Номинальное напряжение (например, "номинальное напряжение 27 В")
    match = re.search(r'номинальное\s*напряжение\s*(\d+[.,]?\d*)\s*[Вв]', text, re.IGNORECASE)
    if match:
        params['nominal_voltage_v'] = float(match.group(1).replace(',', '.'))

    # Степень защиты IP (например, "степень защиты IP68")
    match = re.search(r'степень\s*защиты\s*(IP\d+)', text, re.IGNORECASE)
    if match:
        params['ip_rating'] = match.group(1).upper()

    # Длина (например, "длина 6 м")
    match = re.search(r'длина\s*(\d+[.,]?\d*)\s*м', text, re.IGNORECASE)
    if match:
        params['length_m'] = float(match.group(1).replace(',', '.'))

    # Диаметр (например, "диаметр 160 мм")
    match = re.search(r'диаметр\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['diameter_mm'] = float(match.group(1).replace(',', '.'))

    # Ширина (например, "ширина 5000 мм")
    match = re.search(r'ширина\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['width_mm'] = float(match.group(1).replace(',', '.'))

    # Категория (например, "категория 12")
    match = re.search(r'категория\s*(\d+)', text, re.IGNORECASE)
    if match:
        params['category'] = int(match.group(1))

    # Производительность (например, "производительность 1200 л/ч")
    match = re.search(r'производительность\s*(\d+(?:\.\d+)?)\s*л/ч', text, re.IGNORECASE)
    if match:
        params['performance_l_h'] = float(match.group(1))

    # Мощность (например, "мощность 48 кВт")
    match = re.search(r'мощность\s*(\d+\s*\d+|\d+[.,]?\d*)\s*квт', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['power_kw'] = float(value)

    # Давление пара (например, "давление пара до 0.5 МПа")
    match = re.search(r'давление\s*пара\s*до\s*(\d+[.,]?\d*)\s*М?Па', text, re.IGNORECASE)
    if match:
        params['steam_pressure_mpa'] = float(match.group(1).replace(',', '.'))

    # Температура пара (например, "температура пара до 250")
    match = re.search(r'температура\s*пара\s*до\s*(\d+)', text, re.IGNORECASE)
    if match:
        params['steam_temperature_c'] = int(match.group(1))

    # Диаметр корпуса (например, "диаметр корпуса 100 мм")
    match = re.search(r'диаметр\s*корпуса\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['housing_diameter_mm'] = float(match.group(1).replace(',', '.'))

    # Класс точности (например, "класс точности 1.5")
    match = re.search(r'класс\s*точности\s*(\d+[.,]?\d*)', text, re.IGNORECASE)
    if match:
        params['accuracy_class'] = float(match.group(1).replace(',', '.'))

    # Подача воздуха (например, "подача воздуха 2500 м3/ч")
    match = re.search(r'подача\s*воздуха\s*(\d+(?:\.\d+)?)\s*м3/ч', text, re.IGNORECASE)
    if match:
        params['air_flow_m3_h'] = float(match.group(1))

    # Грузоподъемность (например, "грузоподъемность 250 т")
    match = re.search(r'грузоподъемность\s*(\d+[.,]?\d*)\s*т', text, re.IGNORECASE)
    if match:
        params['load_capacity_t'] = float(match.group(1).replace(',', '.'))

    # Напор (например, "напор 71 м")
    match = re.search(r'напор\s*(\d+[.,]?\d*)\s*м', text, re.IGNORECASE)
    if match:
        params['head_m'] = float(match.group(1).replace(',', '.'))

    # Емкость ковша (например, "емкость ковша 8 м3")
    match = re.search(r'емкость\s*ковша\s*(\d+[.,]?\d*)\s*м3', text, re.IGNORECASE)
    if match:
        params['bucket_capacity_m3'] = float(match.group(1).replace(',', '.'))

    # Мощность нагрева (например, "мощность нагрева клина 1 кВт")
    match = re.search(r'мощность\s*нагрева\s*клина\s*(\d+[.,]?\d*)\с*квт', text, re.IGNORECASE)
    if match:
        params['heating_power_kw'] = float(match.group(1).replace(',', '.'))

    # Рабочее давление (например, "рабочее давление 1 2 мпа")
    match = re.search(r'рабочее\s*давление\s*(\d+\s*\d+|\d+[.,]?\d*)\s*мпа', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['operating_pressure_mpa'] = float(value)

    # Объем бункера (например, "объем бункера 120 л")
    match = re.search(r'объем\s*бункера\s*(\d+[.,]?\d*)\s*л', text, re.IGNORECASE)
    if match:
        params['bunker_volume_l'] = float(match.group(1).replace(',', '.'))

    # Вместимость грузового трюма (например, "вместимость грузового трюма 7000 м3")
    match = re.search(r'вместимость\s*грузового\s*трума\s*(\d+[.,]?\d*)\s*м3', text, re.IGNORECASE)
    if match:
        params['cargo_hold_capacity_m3'] = float(match.group(1).replace(',', '.'))

    # Номинальный сварочный ток (например, "номинальный сварочный ток 250-400 а")
    match = re.search(r'номинальный\s*сварочный\s*ток\s*(\d+\s*\-\s*\d+)\s*[Аа]', text, re.IGNORECASE)
    if match:
        params['nominal_welding_current_a'] = match.group(1).replace(' ', '')  # Сохраняем диапазон как строку "250-400"

    # Производительность по сжатому воздуху (например, "производительность по сжатому воздуху до 8 5 л/мин")
    match = re.search(r'производительность\s*по\s*сжатому\s*воздуху\s*до\s*(\d+\s*\d+|\d+[.,]?\d*)\s*л/мин', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['air_compressor_performance_l_min'] = float(value)

    # Давление (например, "давлением до 40 мпа")
    match = re.search(r'давлени[еи]\s*до\s*(\d+[.,]?\d*)\s*мпа', text, re.IGNORECASE)
    if match:
        params['pressure_mpa'] = float(match.group(1).replace(',', '.'))

    # Объем элегаза (например, "объем элегаза 1000-2000 л")
    match = re.search(r'объем\s*элегаза\s*(\d+\s*\-\s*\d+)\s*л', text, re.IGNORECASE)
    if match:
        params['sf6_volume_l'] = match.group(1).replace(' ', '')  # Сохраняем диапазон как строку "1000-2000"

    # Производительность насоса для откачки элегаза (например, "35 м3/ч")
    match = re.search(r'производительность\s*насоса\s*для\s*откачки\s*элегаза\s*(\d+[.,]?\d*)\s*м3/ч', text, re.IGNORECASE)
    if match:
        params['sf6_pump_performance_m3_h'] = float(match.group(1).replace(',', '.'))

    # Площадь поверхности конденсации (например, "не менее 6 1 м2")
    match = re.search(r'площадь\s*поверхности\s*конденсации\s*не\s*менее\s*(\d+\s*\d+|\d+[.,]?\d*)\s*м2', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['condensation_area_m2'] = float(value)

    # Глубина резки (например, "глубина резки 730 мм")
    match = re.search(r'глубина\s*резки\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['cutting_depth_mm'] = float(match.group(1).replace(',', '.'))

    # Измеряемая глубина (например, "измеряемая глубина до 4 м")
    match = re.search(r'измеряемая\s*глубина\s*до\s*(\d+[.,]?\d*)\s*м', text, re.IGNORECASE)
    if match:
        params['measured_depth_m'] = float(match.group(1).replace(',', '.'))

    # Торкретирование (например, "торкретирования 3 2 м3/ч")
    match = re.search(r'торкретирования\s*(\d+\s*\d+|\d+[.,]?\d*)\s*м3/ч', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['shotcreting_performance_m3_h'] = float(value)

    # Диаметр труб (например, "диаметр труб 125-160 мм")
    match = re.search(r'диаметр\s*труб\s*(\d+\s*\-\s*\d+)\s*мм', text, re.IGNORECASE)
    if match:
        params['pipe_diameter_mm'] = match.group(1).replace(' ', '')  # Сохраняем диапазон как строку "125-160"

    # Мощность двигателя (например, "мощность двигателя до 1 квт")
    match = re.search(r'мощность\s*двигателя\s*до\s*(\d+[.,]?\d*)\s*квт', text, re.IGNORECASE)
    if match:
        params['engine_power_kw'] = float(match.group(1).replace(',', '.'))

    # Количество секций (например, "количество секций 2")
    match = re.search(r'количество\s*секций\s*(\d+)', text, re.IGNORECASE)
    if match:
        params['sections_count'] = int(match.group(1))

    # Мощность одной секции (например, "мощность одной секции 300 квт")
    match = re.search(r'мощность\s*одной\s*секции\s*(\d+[.,]?\d*)\s*квт', text, re.IGNORECASE)
    if match:
        params['section_power_kw'] = float(match.group(1).replace(',', '.'))

    # Диапазон диаметров (например, "диаметром от 630 до 1200 мм")
    match = re.search(r'диаметром\s*от\s*(\d+)\s*до\s*(\d+)\s*мм', text, re.IGNORECASE)
    if match:
        params['diameter_range_mm'] = f"{match.group(1)}-{match.group(2)}"

    # Мощность привода фрезы (например, "мощность привода фрезы 6600 квт")
    match = re.search(r'мощность\s*привода\s*фрезы\s*(\d+[.,]?\d*)\s*квт', text, re.IGNORECASE)
    if match:
        params['milling_power_kw'] = float(match.group(1).replace(',', '.'))

    # Поиск материалов
    materials = ['сталь', 'пластик', 'оцинкованная', 'латунь', 'асбест', 'хризотил', 'цемент']
    for mat in materials:
        if re.search(r'\b' + re.escape(mat) + r'\b', text, re.IGNORECASE):
            params['material'] = mat
            break

    # Горизонтальная поляризация
    if re.search(r'горизонтальной\s*поляризацией', text, re.IGNORECASE):
        params['polarization'] = 'горизонтальная'

    # Линейная ортогональная поляризация
    if re.search(r'линейной\s*ортогональной\s*поляризацией', text, re.IGNORECASE):
        params['polarization'] = 'линейная ортогональная'


    # Допустимое напряжение до (например, "до 7 2 кв")
    match = re.search(r'допустимое\s*напряжение\s*до\s*(\d+\s*\d+|\d+[.,]?\d*)\s*к[Вв]', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['max_voltage_kv'] = float(value)

    # Номинальный разрядный ток (например, "номинальный разрядный ток 10 ка")
    match = re.search(r'номинальный\s*разрядный\s*ток\s*(\d+[.,]?\d*)\s*к[Аа]', text, re.IGNORECASE)
    if match:
        params['nominal_discharge_current_ka'] = float(match.group(1).replace(',', '.'))

    # Класс пропускной способности
    match = re.search(r'класс\s*пропускной\s*способности\s*(\d+)', text, re.IGNORECASE)
    if match:
        params['surge_class'] = int(match.group(1))

    # Емкость аккумуляторной батареи (например, "емкость аккумуляторной батареи 17 а")
    match = re.search(r'емкость\s*аккумуляторной\s*батареи\s*(\d+[.,]?\d*)\s*[Аа]\*ч', text, re.IGNORECASE)
    if match:
        params['battery_capacity_ah'] = float(match.group(1).replace(',', '.'))

    # Выходное напряжение (например, "13 3-13 8 в")
    match = re.search(r'выходное\s*напряжение.*?(\d+\s*\d+\-\d+\s*\d+|\d+[.,]?\d*\-\d+[.,]?\d*)\s*[Вв]', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['output_voltage_v'] = value

    # Максимальный ток (например, "максимальный ток 50 а")
    match = re.search(r'максимальный\s*ток\s*(\d+[.,]?\d*)\s*[Аа]', text, re.IGNORECASE)
    if match:
        params['max_current_a'] = float(match.group(1).replace(',', '.'))

    # Диапазон измерения температуры (например, "от -50 °c до +50")
    match = re.search(r'диапазон\s*измерения\s*от\s*(-?\d+)\s*[°\s]?[Cc]\s*до\s*(\d+)', text, re.IGNORECASE)
    if match:
        params['temperature_range_c'] = f"{match.group(1)}..{match.group(2)}"

    # Электромагнитный
    if re.search(r'электромагнитный', text, re.IGNORECASE):
        params['actuator_type'] = 'электромагнитный'

    # Напряжение переменное 660 В
    match = re.search(r'напряжение\s*переменное\s*(\d+[.,]?\d*)\s*[Вв]', text, re.IGNORECASE)
    if match:
        params['ac_voltage_v'] = float(match.group(1).replace(',', '.'))

    # Диаметр апертуры (например, "диаметр апертуры 300 мм")
    match = re.search(r'диаметр\s*апертуры\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['aperture_diameter_mm'] = float(match.group(1).replace(',', '.'))

    # Дорожный транспортный светодиодный
    if re.search(r'дорожный\s*транспортный\s*светодиодный', text, re.IGNORECASE):
        params['light_type'] = 'дорожный транспортный светодиодный'

    # Напряжение питания переменного тока (например, "10 0-12 0")
    match = re.search(r'напряжение\s*питания\s*переменного\s*тока\s*(\d+\s*\d+\-\d+\s*\d+|\d+[.,]?\d*\-\d+[.,]?\d*)', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['ac_power_voltage_v'] = value

    # Потребляемая мощность (например, "не более 15 вт")
    match = re.search(r'потребляемая\s*мощность\s*не\s*более\s*(\d+[.,]?\d*)\s*вт', text, re.IGNORECASE)
    if match:
        params['power_consumption_w'] = float(match.group(1).replace(',', '.'))

    # Цвета (красный, желтый, синий)
    colors = ['красный', 'желтый', 'синий']
    for color in colors:
        if re.search(r'\b' + re.escape(color) + r'\b', text, re.IGNORECASE):
            params['color'] = color
            break

    # ВВП-16
    if re.search(r'ввп-16', text, re.IGNORECASE):
        params['model'] = 'ВВП-16'


    # Продольная перфорация (например, "25 %")
    match = re.search(r'продольная\s*перфорация\s*(\d+[.,]?\d*)\s*%', text, re.IGNORECASE)
    if match:
        params['perforation_percent'] = float(match.group(1).replace(',', '.'))

    # Толщина стенки (например, "11 5 мм")
    match = re.search(r'толщина\s*стенки\s*(\d+\s*\d+|\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['wall_thickness_mm'] = float(value)


    # Удерживающая способность (например, "400 кДж")
    match = re.search(r'удерживающая\s*способность\s*(\d+[.,]?\d*)\s*кдж', text, re.IGNORECASE)
    if match:
        params['holding_capacity_kj'] = float(match.group(1).replace(',', '.'))

    # Динамический прогиб до (например, "до 1100 мм")
    match = re.search(r'динамический\s*прогиб\s*до\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['dynamic_deflection_mm'] = float(match.group(1).replace(',', '.'))

    # Длина секции и высота (например, "4000 мм высота 3000 мм")
    match = re.search(r'длина\s*секции\s*(\d+)\s*мм\s*высота\s*(\d+)\s*мм', text, re.IGNORECASE)
    if match:
        params['section_dimensions_mm'] = f"{match.group(1)}x{match.group(2)}"

    # Ширина профиля (например, "160 мм")
    match = re.search(r'ширина\s*профиля\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['profile_width_mm'] = float(match.group(1).replace(',', '.'))

    # Тип (например, "8.4.9-8.4.14")
    match = re.search(r'тип\s*((?:\d+\.){2}\d+\-(?:\d+\.){2}\d+)', text, re.IGNORECASE)
    if match:
        params['equipment_type'] = match.group(1)



    # Трубы железобетонные безнапорные
    if re.search(r'трубы\s*железобетонные\s*безнапорные', text, re.IGNORECASE):
        params['pipe_type'] = 'железобетонные безнапорные'

    # Трубы железобетонные безнапорные раструбные
    if re.search(r'трубы\s*железобетонные\s*безнапорные\s*раструбные', text, re.IGNORECASE):
        params['pipe_type'] = 'железобетонные безнапорные раструбные'

    # Класс бетона (например, "В40")
    match = re.search(r'класс\s*бетон\s*(В\d+)', text, re.IGNORECASE)
    if match:
        params['concrete_class'] = match.group(1)

    # Объем бетона (например, "1 1")
    match = re.search(r'объем\s*бетона\s*(\d+\s*\d+|\d+[.,]?\d*)', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['concrete_volume'] = float(value)

    # Наибольшее измерение (например, "от 301 до 400 мм")
    match = re.search(r'наибольшее\s*измерение\s*от\s*(\d+)\s*до\s*(\d+)\s*мм', text, re.IGNORECASE)
    if match:
        params['max_measurement_mm'] = f"{match.group(1)}-{match.group(2)}"

    # Лицевые размеры (например, "250х120х140 мм")
    match = re.search(r'лицевой\s*размеры\s*(\d+)х(\d+)х(\d+)\s*мм', text, re.IGNORECASE)
    if match:
        params['face_dimensions_mm'] = f"{match.group(1)}x{match.group(2)}x{match.group(3)}"

    # Марка М200
    if re.search(r'марка\s*М200', text, re.IGNORECASE):
        params['brand'] = 'М200'

    # Площадь (например, "2 05 м2")
    match = re.search(r'площадь\s*(\d+\s*\d+|\d+[.,]?\d*)\s*м2', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['area_m2'] = float(value)

    # Диаметр трубы (например, "159 мм")
    match = re.search(r'диаметр\s*трубы\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['pipe_diameter_mm'] = float(match.group(1).replace(',', '.'))

    # Толщина (например, "4 мм")
    match = re.search(r'толщина\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['thickness_mm'] = float(match.group(1).replace(',', '.'))

    # Размер фланца (например, "250 мм")
    match = re.search(r'размер\s*фланца\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['flange_size_mm'] = float(match.group(1).replace(',', '.'))

    # Количество отверстий фланца (например, "4 шт")
    match = re.search(r'количество\s*отверстий\s*фланца\s*(\d+)\s*шт', text, re.IGNORECASE)
    if match:
        params['flange_holes_count'] = int(match.group(1))

    # Диаметр отверстий крепежных элементов (например, "20 мм")
    match = re.search(r'диаметр\s*отверстий\s*крепежных\s*элементов\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['fastener_hole_diameter_mm'] = float(match.group(1).replace(',', '.'))

    # Высота закладной (например, "1300 мм")
    match = re.search(r'высота\s*закладной\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['embedment_height_mm'] = float(match.group(1).replace(',', '.'))

    # Вылет (например, "1000 мм")
    match = re.search(r'вылет\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['overhang_mm'] = float(match.group(1).replace(',', '.'))

    # Угол между посадочными местами (например, "120°")
    match = re.search(r'угол\s*между\s*посадочными\s*местами\s*(\d+[.,]?\d*)\s*°', text, re.IGNORECASE)
    if match:
        params['mounting_angle_deg'] = float(match.group(1).replace(',', '.'))

    # Марка ОТ-251
    if re.search(r'марка\s*ОТ-251', text, re.IGNORECASE):
        params['brand'] = 'ОТ-251'

    # Тип покрытия (например, "полиэстер")
    if re.search(r'тип\s*покрытия\s*полиэстер', text, re.IGNORECASE):
        params['coating_type'] = 'полиэстер'

    # Толщина стали (например, "0 6 мм")
    match = re.search(r'олщина\s*стали\s*(\d+\s*\d+|\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['steel_thickness_mm'] = float(value)

    # Масса (например, "от 100 до 150 кг")
    match = re.search(r'масса\s*от\s*(\d+)\s*до\s*(\d+)\s*кг', text, re.IGNORECASE)
    if match:
        params['mass_kg'] = f"{match.group(1)}-{match.group(2)}"

    # Класс напряжения (например, "35 кВ")
    match = re.search(r'класс\s*напряжения\s*(\d+[.,]?\d*)\s*к[Вв]', text, re.IGNORECASE)
    if match:
        params['voltage_class_kv'] = float(match.group(1).replace(',', '.'))

    # Высота надземной части опоры (например, "10000 мм")
    match = re.search(r'высота\s*надземной\s*части\s*опоры\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['pole_height_above_ground_mm'] = float(match.group(1).replace(',', '.'))

    # Высота закладного элемента фундамента (например, "2000 мм")
    match = re.search(r'высота\s*закладного\s*элемента\s*фундамента\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['foundation_embedment_height_mm'] = float(match.group(1).replace(',', '.'))

    # Диаметр в нижней части опоры (например, "232 мм")
    match = re.search(r'диаметр\s*в\s*нижней\s*части\s*опоры\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['pole_lower_diameter_mm'] = float(match.group(1).replace(',', '.'))

    # Диаметр в верхней части опоры (например, "100 мм")
    match = re.search(r'диаметр\s*в\s*верхней\s*части\s*опоры\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['pole_upper_diameter_mm'] = float(match.group(1).replace(',', '.'))

    # Размеры фланца (например, "400х400 мм")
    match = re.search(r'размеры\s*фланца\s*(\d+)х(\d+)\s*мм', text, re.IGNORECASE)
    if match:
        params['flange_dimensions_mm'] = f"{match.group(1)}x{match.group(2)}"

    # Нагрузка в верхней части опоры (например, "300 кг")
    match = re.search(r'нагрузкой\s*в\s*верхней\s*части\s*опоры\s*(\d+[.,]?\d*)\s*кг', text, re.IGNORECASE)
    if match:
        params['pole_upper_load_kg'] = float(match.group(1).replace(',', '.'))

    # Диаметр лопасти (например, "300 мм")
    match = re.search(r'диаметр\s*лопасти\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['blade_diameter_mm'] = float(match.group(1).replace(',', '.'))

    # Диаметр ствола и толщина стенки (например, "159 мм толщина стенки 4")
    match = re.search(r'диаметр\s*ствола\s*(\d+[.,]?\d*)\s*мм\s*толщина\s*стенки\s*(\d+[.,]?\d*)', text, re.IGNORECASE)
    if match:
        params['trunk_diameter_wall_thickness_mm'] = f"{match.group(1)}x{match.group(2)}"

    # Длина хомута (например, "1414 мм")
    match = re.search(r'длина\s*хомута\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['clamp_length_mm'] = float(match.group(1).replace(',', '.'))

    # Маркировочная группа (например, "1570-1770 н/мм2")
    match = re.search(r'маркировочная\s*группа\s*(\d+\s*\d+\-\d+\s*\d+|\d+[.,]?\d*\-\d+[.,]?\d*)\s*н/мм2', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['marking_group_n_mm2'] = value

    # Сторона квадрата (например, "9 мм")
    match = re.search(r'сторона\s*квадрата\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['square_side_mm'] = float(match.group(1).replace(',', '.'))

    # Толщина полки (например, "22 мм")
    match = re.search(r'толщина\s*полки\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['flange_thickness_mm'] = float(match.group(1).replace(',', '.'))

    # Волокна толщиной (например, "70 мм")
    match = re.search(r'волокна\s*толщиной\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['fiber_thickness_mm'] = float(match.group(1).replace(',', '.'))

    # Площадь (например, "от 3 51 до 4 м2")
    match = re.search(r'площадь\s*от\s*(\d+\s*\d+|\d+[.,]?\d*)\s*до\s*(\d+\s*\d+|\d+[.,]?\d*)\s*м2', text, re.IGNORECASE)
    if match:
        value = f"{match.group(1).replace(' ', '').replace(',', '.')}..{match.group(2).replace(' ', '').replace(',', '.')}"
        params['area_range_m2'] = value

    # Класс износостойкости (например, "31")
    match = re.search(r'класс\s*износостойкости\s*(\d+)', text, re.IGNORECASE)
    if match:
        params['abrasion_class'] = int(match.group(1))

    # Диаметр желоба (например, "185 мм")
    match = re.search(r'диаметр\s*желоба\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['gutter_diameter_mm'] = float(match.group(1).replace(',', '.'))

    # Диаметр трубы и длина (например, "100 мм длина трубы 1000 мм")
    match = re.search(r'диаметр\s*трубы\s*(\d+[.,]?\d*)\s*мм\s*длина\s*трубы\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['pipe_diameter_length_mm'] = f"{match.group(1)}x{match.group(2)}"

    # Гибкость не ниже (например, "-5 °C")
    match = re.search(r'гибкость\s*не\s*ниже\s*(?:\+|\-)?\s*(\d+)[°\s]?[Cc]', text, re.IGNORECASE)
    if match:
        params['flexibility_c'] = int(match.group(1))

    # Прочность не менее (например, "350 н")
    match = re.search(r'прочность\s*не\s*менее\s*(\d+[.,]?\d*)\s*[Нн]', text, re.IGNORECASE)
    if match:
        params['strength_n'] = float(match.group(1).replace(',', '.'))

    # Высота выступов (например, "8 5 мм")
    match = re.search(r'высота\s*выступов\s*(\d+\s*\d+|\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['bump_height_mm'] = float(value)

    # Предел прочности на сжатие (например, "200 кн/м2")
    match = re.search(r'предел\s*прочности\s*на\s*сжатие\s*(\d+[.,]?\d*)\s*кн/м2', text, re.IGNORECASE)
    if match:
        params['compression_strength_kn_m2'] = float(match.group(1).replace(',', '.'))

    # Толщина полотна (например, "2 5 мм")
    match = re.search(r'толщина\s*полотна\s*(\d+\s*\d+|\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['sheet_thickness_mm'] = float(value)

    # Прочность при изгибе (например, "не менее 0 15 мпа")
    match = re.search(r'прочность\s*при\s*изгибе\s*не\s*менее\s*(\d+\s*\d+|\d+[.,]?\d*)\s*мпа', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['bending_strength_mpa'] = float(value)

    # Внутренний диаметр (например, "108 мм")
    match = re.search(r'внутренний\s*диаметр\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['inner_diameter_mm'] = float(match.group(1).replace(',', '.'))

    # Плотность (например, "44±10 кг/м3")
    match = re.search(r'плотность\s*(\d+[.,]?\d*)±(\d+[.,]?\d*)\s*кг/м3', text, re.IGNORECASE)
    if match:
        params['density_kg_m3'] = f"{match.group(1)}±{match.group(2)}"

    # Температура применения (например, "от -200 до +110 °C")
    match = re.search(r'температура\s*применения\s*от\s*(-?\d+)\s*до\s*(\+?\d+)\s*°[Cc]', text, re.IGNORECASE)
    if match:
        params['operating_temp_c'] = f"{match.group(1)}..{match.group(2)}"

    # Группа горючести (например, "Г1")
    match = re.search(r'группа\s*горючести\s*([Гг][1234])', text, re.IGNORECASE)
    if match:
        params['flammability_group'] = match.group(1).upper()

    # Максимальная температура применения (например, "+650 °C")
    match = re.search(r'максимальная\s*температура\s*применения\s*(?:\+|\-)(\d+)\s*°[Cc]', text, re.IGNORECASE)
    if match:
        params['max_temp_c'] = int(match.group(1))

    # Теплопроводность (например, "не более 0 036/0 039 Вт/м*К")
    match = re.search(r'теплопроводность.*?(\d+\s*\d+|\d+[.,]?\d*)/(\d+\s*\d+|\d+[.,]?\d*)\s*вт/м\*к', text, re.IGNORECASE)
    if match:
        value = f"{match.group(1).replace(' ', '.')}..{match.group(2).replace(' ', '.')}"
        params['thermal_conductivity_w_mk'] = value

    # Расход (например, "2 2 кг/м2")
    match = re.search(r'расход\s*(\d+\s*\d+|\d+[.,]?\d*)\s*кг/м2', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['consumption_kg_m2'] = float(value)

    # Плотность (например, "0 830-0 880 г/см3")
    match = re.search(r'плотность\s*(\d+\s*\d+\-\d+\s*\d+|\d+[.,]?\d*\-\d+[.,]?\d*)\s*г/см3', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['density_g_cm3'] = value

    # Высота ограждения и длина (например, "800 мм длина ограждения 3000 мм")
    match = re.search(r'высота\s*ограждения\s*(\d+[.,]?\d*)\s*мм.*?длина\s*ограждения\s*(\d+[.,]?\d*)\s*мм', text, re.IGNORECASE)
    if match:
        params['fence_dimensions_mm'] = f"{match.group(1)}x{match.group(2)}"

    # Высота (например, "1 5-2 0 м")
    match = re.search(r'высота\s*(\d+\s*\d+|\d+[.,]?\d*)\s*\-\s*(\d+\s*\d+|\d+[.,]?\d*)\s*м', text, re.IGNORECASE)
    if match:
        value = f"{match.group(1).replace(' ', '.')}..{match.group(2).replace(' ', '.')}"
        params['height_range_m'] = value

    # Номинальное давление (например, "1 6 мпа")
    match = re.search(r'номинальное\s*давление\s*(\d+\s*\d+|\d+[.,]?\d*)\s*мпа', text, re.IGNORECASE)
    if match:
        value = match.group(1).replace(' ', '').replace(',', '.')
        params['nominal_pressure_mpa'] = float(value)

    return params

### Мы после параметризации be like:
<img src="photo_2025-06-01 20.52.13.jpeg" width="350">

###Обработка листов Excel: нормализация наименований, токенизация, извлечение параметров и сохранение результата

In [ ]:
dfs = pd.read_excel('processed_by_category.xlsx', sheet_name=['Материалы', 'Оборудование', 'Механизмы'])

# Получаем DataFrame для каждого листа
df_materials = dfs['Материалы']
df_equipment = dfs['Оборудование']
df_machines = dfs['Механизмы']

# Применение обработки к каждому DataFrame
df_materials['Наименование'] = df_materials['Наименование'].apply(normalize_text)
df_materials['tokens'] = df_materials['Наименование'].apply(tokenize)
params_df_materials = df_materials['Наименование'].apply(extract_params).apply(pd.Series)

df_equipment['Наименование'] = df_equipment['Наименование'].apply(normalize_text)
df_equipment['tokens'] = df_equipment['Наименование'].apply(tokenize)
params_df_equipment = df_equipment['Наименование'].apply(extract_params).apply(pd.Series)

df_machines['Наименование'] = df_machines['Наименование'].apply(normalize_text)
df_machines['tokens'] = df_machines['Наименование'].apply(tokenize)
params_df_machines = df_machines['Наименование'].apply(extract_params).apply(pd.Series)

# Объединение DataFrame с параметрами
df_materials = pd.concat([df_materials, params_df_materials], axis=1)
df_equipment = pd.concat([df_equipment, params_df_equipment], axis=1)
df_machines = pd.concat([df_machines, params_df_machines], axis=1)

# Сохранение в новый файл
with pd.ExcelWriter('processed_with_params.xlsx', engine='openpyxl') as writer:
    df_materials.to_excel(writer, sheet_name='Материалы', index=False)
    df_equipment.to_excel(writer, sheet_name='Оборудование', index=False)
    df_machines.to_excel(writer, sheet_name='Механизмы', index=False)